# imports

In [2]:
import duckdb
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import os

# config

In [1]:
ASSET      = "AAPL"
INTERVAL   = "1h"
HORIZON    = 4

# connection and load

In [3]:
db_path = '../../database/financial_data.duckdb'
conn = duckdb.connect(db_path)
df = conn.execute(f"""
    SELECT * FROM gold_ml_features
    WHERE asset_symbol = '{ASSET}'
    AND interval = '{INTERVAL}'
    ORDER BY date ASC
""").df()
df['date'] = pd.to_datetime(df['date'])
df.set_index('date', inplace=True)
print(f"Loaded {len(df)} rows for {ASSET} {INTERVAL}")

Loaded 3473 rows for AAPL 1h


# drop metadata cols

In [4]:
metadata_cols = ['asset_symbol', 'asset_class', 'exchange', 'interval']
df = df.drop(columns=metadata_cols, errors='ignore')
print(f"Columns remaining: {len(df.columns)}")

Columns remaining: 44


# 4hr target , return over 4 periods shift(-4)

In [5]:
df['target_4h'] = df['close'].pct_change(HORIZON).shift(-HORIZON)
df = df.dropna(subset=['target_4h'])
df['target_direction'] = (df['target_4h'] > 0).astype(int)
balance = df['target_direction'].value_counts(normalize=True) * 100
print("Class Balance (4h target):")
print(balance)
print(f"\nRows remaining after target creation: {len(df)}")

Class Balance (4h target):
target_direction
1    54.597867
0    45.402133
Name: proportion, dtype: float64

Rows remaining after target creation: 3469


# stationary conversion

In [6]:
df['sma_7_dist']   = df['close'] / df['sma_7']   - 1
df['sma_30_dist']  = df['close'] / df['sma_30']  - 1
df['sma_50_dist']  = df['close'] / df['sma_50']  - 1
df['sma_100_dist'] = df['close'] / df['sma_100'] - 1
df['sma_200_dist'] = df['close'] / df['sma_200'] - 1
df['ema_12_dist']  = df['close'] / df['ema_12']  - 1
df['ema_26_dist']  = df['close'] / df['ema_26']  - 1
df['ema_50_dist']  = df['close'] / df['ema_50']  - 1
df['ema_200_dist'] = df['close'] / df['ema_200'] - 1
df['vwap_dist']    = df['close'] / df['vwap']    - 1

In [7]:
df['macd_pct']       = df['macd'] / df['close']
df['macd_sig_pct']   = df['macd_signal'] / df['close']
df['macd_hist_pct']  = df['macd_histogram'] / df['close']
df['atr_pct']        = df['atr_14'] / df['close']
df['volatility_pct'] = df['daily_volatility'] / df['close']

# drop non stationary

In [9]:
non_stationary_cols = [
    'open', 'high', 'low', 'close',
    'sma_7', 'sma_30', 'sma_50', 'sma_100', 'sma_200',
    'ema_12', 'ema_26', 'ema_50', 'ema_200',
    'vwap', 'prev_close', 'prev_high', 'prev_low',
    'macd', 'macd_signal', 'macd_histogram',
    'atr_14', 'daily_volatility',
    'bb_upper', 'bb_middle', 'bb_lower', 'bb_width',
    'volume', 'prev_volume', 'volume_sma_20',
    'obv'
]
df = df.drop(columns=non_stationary_cols, errors='ignore')
df = df.dropna()
print(f"Final shape : {df.shape}")
df.head(3)

Final shape : (3469, 31)


,rsi_14,roc_10,roc_20,stoch_k,stoch_d,bb_percentage,volume_ratio,returns_1p,returns_5p,returns_10p,...,ema_12_dist,ema_26_dist,ema_50_dist,ema_200_dist,vwap_dist,macd_pct,macd_sig_pct,macd_hist_pct,atr_pct,volatility_pct
date,,,,,,,,,,,,,,,,,,,,,
2024-05-13 16:30:00,73.207505,1.885100,2.175111,94.345968,92.559982,1.032018,1.247968,0.001979,0.020327,0.018851,...,0.011217,0.016993,0.027924,0.066526,0.011887,0.005617,0.004401,0.001216,0.005006,0.003643
2024-05-13 17:30:00,73.117205,2.167660,2.055876,93.778076,94.439465,0.956074,0.837094,-0.000046,0.019946,0.021677,...,0.009436,0.015672,0.026755,0.065773,0.011108,0.006083,0.004737,0.001345,0.004870,0.003107
2024-05-13 18:30:00,74.472907,2.466507,2.325650,97.784559,95.302868,0.947359,0.990380,0.001821,0.007006,0.024665,...,0.009523,0.016202,0.027471,0.066994,0.012059,0.006511,0.005085,0.001426,0.004721,0.002888


# chronological split 80/20

In [10]:
split_idx = int(len(df) * 0.80)
train_df = df.iloc[:split_idx].copy()
test_df  = df.iloc[split_idx:].copy()
print(f"Training: {train_df.index.min()} to {train_df.index.max()} (Rows: {len(train_df)})")
print(f"Testing : {test_df.index.min()} to {test_df.index.max()} (Rows: {len(test_df)})")

Training: 2024-05-13 16:30:00 to 2025-12-15 18:30:00 (Rows: 2775)
Testing : 2025-12-15 19:30:00 to 2026-05-11 15:30:00 (Rows: 694)


# scale features

In [11]:
features = [col for col in train_df.columns if col not in ['target_4h', 'target_direction']]
scaler = StandardScaler()
train_df[features] = scaler.fit_transform(train_df[features])
test_df[features]  = scaler.transform(test_df[features])
print(f"Scaled {len(features)} features")

Scaled 29 features


# saving into parquets

In [12]:
os.makedirs('../../data/processed', exist_ok=True)
train_df.to_parquet(f'../../data/processed/train_{ASSET.lower()}_{INTERVAL}_4h_target.parquet')
test_df.to_parquet(f'../../data/processed/test_{ASSET.lower()}_{INTERVAL}_4h_target.parquet')
print(f"Saved: train_{ASSET.lower()}_{INTERVAL}_4h_target.parquet")
print(f"Saved: test_{ASSET.lower()}_{INTERVAL}_4h_target.parquet")

Saved: train_aapl_1h_4h_target.parquet
Saved: test_aapl_1h_4h_target.parquet
